# R4Cs – Monthly Mean NDVI Composites from Sentinel-2
**Project:** Restore4Cs (Restoring Soil Carbon in European Wetlands) – WP6  
**Author:** Nina Bègue  
**Last update:** 11/2025  

---

## What this script does

This notebook computes **monthly multi-annual mean NDVI composites** (one image per calendar month, January to December) for a given wetland site, using Sentinel-2 Surface Reflectance imagery (2017–2024) from Google Earth Engine (GEE).

For each month, the composite is the pixel-wise mean NDVI across all cloud-free observations over the full 2017–2024 period. Pixels with no valid observations for a given month are filled with the annual mean NDVI (computed over all months and years).

**Output:** 12 GeoTIFF files named `{site}_vegetation_01.tif` to `{site}_vegetation_12.tif`, exported to Google Drive.

---

## Prerequisites

### 1. Google Earth Engine account
You need a GEE account with an active project. Replace `'ee-your-project'` in the authentication cell below with your own GEE project ID.

### 2. Google Drive folder structure
Create the following folder structure in **your Google Drive** (under `My Drive`):

```
My Drive/
└── R4Cs_WP6_layers/
    ├── Camargue/
    │   └── Camargue.gpkg          ← GeoPackage of the site boundary
    ├── Valencia/
    │   └── Valencia.gpkg
    ├── Curonian_Lagoon/
    │   └── Curonian_Lagoon.gpkg
    ├── Danube_delta/
    │   └── Danube_delta.gpkg
    └── ...
```

Each `.gpkg` file must contain the **polygon(s) defining the study area boundary** for that site. The file can contain one or several polygons (tiles); the script processes them all.

### 3. Site selection
In the **"Site selection"** cell below, set the corresponding variable to `True` for the site you want to process. Only one site should be set to `True` at a time.

| Variable | Site | Nb_S2_tile | dist_th` (edge mask, px) |
|---|---|---|---|
| `do_FR` | Camargue (France) | 4 | 10 |
| `do_ES` | Valencia (Spain) | 4 | 5 |
| `do_CU` | Curonian Lagoon (Lithuania) | 2 | 5 |
| `do_DA` | Danube Delta (Romania) | 5 | 20 |
| `do_DU` | DU site (Netherlands) | ? | 5 |
| `do_AV` | Aveiro (Portugal) | ? | 10 |

> **`dist_th`** controls how many pixels are masked near the edge of each Sentinel-2 tile (to remove unreliable border pixels). Higher values = more aggressive edge removal. Higher dist_th when more S2 tile to cover the site (so more edge artefacts).

---

## Processing pipeline

```
S2_SR_HARMONIZED (2017–2024)
    → Cloud filtering (S2_CLOUD_PROBABILITY < 20% + SCL mask)
    → Edge pixel removal (feather mask, dist_th pixels)
    → NDVI computation: (B8 - B4) / (B8 + B4)
    → Monthly mean composite (per calendar month, all years)
    → Missing pixels filled with annual mean composite
    → Export to Google Drive (GeoTIFF, 10 m)
```

## Step 0 – Mount Google Drive and import libraries

In [ ]:
# Mount Google Drive (required to read the .gpkg boundary files)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import ee
import geemap
import geopandas as gpd
import os
import re

## Step 1 – Authenticate and initialise Google Earth Engine

In [ ]:
# Replace 'ee-your-project' with your own GEE project ID
GEE_PROJECT = 'ee-your-project'  # ← EDIT THIS

try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)

## Step 2 – Site selection and export options

Set **one** site variable to `True`. All others must remain `False`.  
Also choose what to export using the `do_*` flags at the bottom.

In [ ]:
# ── Site selection (set only ONE to True) ────────────────────────────────────
do_FR     = False  # Camargue, France
do_ES     = False  # Valencia, Spain
do_CU     = True   # Curonian Lagoon
do_DA     = False  # Danube Delta
do_DU     = False  # DU site
do_VA = False      # Aveiro, Portugal

# ── Export options ────────────────────────────────────────────────────────────
# Set to True to export the corresponding product to Google Drive
do_month     = True   # Monthly mean NDVI composites (12 GeoTIFFs) ← main output
do_mean_NDVI = False  # Full-period mean NDVI (single GeoTIFF)
do_mean_MNDWI= False  # Full-period mean MNDWI (single GeoTIFF)
do_NDVI      = False  # All individual NDVI images (large export)
do_images    = False  # All individual S2 images (very large export)

## Step 3 – Site parameters (automatic, do not edit)

In [ ]:
# Parameters are set automatically based on the site selected above.
# dist_th : number of pixels masked near tile edges (feather mask threshold).
# A higher value removes more edge pixels, reducing tile-boundary artefacts.

if do_FR:
    pilote  = 'Camargue'
    dist_th = 10
elif do_ES:
    pilote  = 'Valencia'
    dist_th = 5
elif do_CU:
    pilote  = 'Curonian_Lagoon'
    dist_th = 50
elif do_DA:
    pilote  = 'Danube_delta'
    dist_th = 10
elif do_DU:
    pilote  = 'DU'
    dist_th = 5
elif do_RA:
  pilote='Aveiro'
  dist_th = 10
else:
    raise ValueError("No site selected. Set one of the do_* variables to True.")

# Google Drive folder that contains the .gpkg files and receives the exports
DRIVE_FOLDER = 'R4Cs_WP6_layers'
path_to_data  = f"/content/drive/MyDrive/{DRIVE_FOLDER}/{pilote}/"

print(f"Site     : {pilote}")
print(f"dist_th  : {dist_th} px")
print(f"Data path: {path_to_data}")

## Step 4 – Load site boundary (.gpkg)

In [ ]:
# List all .gpkg files in the site folder.
# Each .gpkg defines one spatial tile (polygon) of the study area.
# The script processes them all and exports one set of composites per tile.
tiles = [f for f in os.listdir(path_to_data) if f.endswith(".gpkg")]
print(f"Found {len(tiles)} tile(s): {tiles}")

In [ ]:
# Helper functions
def remove_special_characters(text):
    """Remove non-alphanumeric characters (used for GEE task naming)."""
    return re.sub(r'[^a-zA-Z0-9]', '', text)

def polygon_to_ee(polygon):
    """Convert a Shapely polygon exterior to a GEE-compatible coordinate list."""
    return [[[x, y] for x, y, *_ in polygon.exterior.coords]]

In [ ]:
# Read each .gpkg and convert polygons to GEE FeatureCollections.
# Two geometries are created per polygon:
#   - geom_buff : slightly buffered (0.003°) → used to load S2 images (avoids edge gaps)
#   - geom      : lightly buffered (0.001°) → used as the export region

features_buff = []  # buffered geometries (image loading)
features      = []  # export geometries

for file in tiles:
    gdf = gpd.read_file(path_to_data + file).to_crs(epsg=4326)
    for _, row in gdf.iterrows():
        geom      = row.geometry.buffer(0.001)
        geom_buff = geom.buffer(0.003)

        coords_buff = ([polygon_to_ee(p) for p in geom_buff.geoms]
                       if geom_buff.geom_type == 'MultiPolygon'
                       else [polygon_to_ee(geom_buff)])
        coords      = ([polygon_to_ee(p) for p in geom.geoms]
                       if geom.geom_type == 'MultiPolygon'
                       else [polygon_to_ee(geom)])

        tile_id = tiles[0].split('.')[0]
        features_buff.append(ee.Feature(ee.Geometry.MultiPolygon(coords_buff)).set({"ID": tile_id}))
        features.append(ee.Feature(ee.Geometry.MultiPolygon(coords)).set({"ID": tile_id}))

geometry           = ee.FeatureCollection(features_buff)  # for image loading
geometry_to_export = ee.FeatureCollection(features)        # for export clipping
print("Geometries loaded successfully.")

## Step 5 – Visualise site boundary on map *(optional)*

In [ ]:
# Interactive map to verify that the site boundary is correctly loaded.
# This cell is optional and can be skipped.

# map_view = geemap.Map(ee_initialize=False)
# map_view.setControlVisibility()
# map_view.centerObject(geometry.first(), 8)
# map_view.addLayer(geometry,           {}, 'Site boundary (buffered)')
# map_view.addLayer(geometry_to_export, {}, 'Site boundary (export)')
# map_view

## Step 6 – Load Sentinel-2 image collection

In [ ]:
# Time period covered
startDate = '2017-01-01'  # Sentinel-2 SR available from early 2017 on GEE
endDate   = '2024-12-31'

# Sentinel-2 bands to load.
# SCL (Scene Classification Layer) is included for cloud/shadow/snow masking.
bands = ['B2', 'B3', 'B4', 'B8', 'B8A', 'B11', 'B12', 'SCL']

# Load the Sentinel-2 SR Harmonized collection, filtered by date and site bounds.
# Processing is tile-by-tile (see Step 4) to avoid GEE memory limits.
colS2A = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterDate(startDate, endDate)
            .filterBounds(geometry)
            .select(bands))

# Retrieve the native projection from the first image (used for all reprojections)
dst_crs = colS2A.first().select('B2').projection()
print("Collection loaded. Native CRS retrieved.")

## Step 7 – Cloud masking

Two complementary cloud masks are applied:
1. **S2_CLOUD_PROBABILITY** (image-level filter + pixel-level mask): removes images and pixels with > 20% cloud probability.
2. **SCL band** (Scene Classification Layer): removes pixels classified as saturated, cloud shadow, medium/high clouds, thin cirrus, or snow.

In [ ]:
# ── 7a. Load the cloud probability collection and join it to S2 ──────────────
colS2_cloud = (ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")
                 .filterDate(startDate, endDate)
                 .filterBounds(geometry))

# Join S2_SR and S2_CLOUD_PROBABILITY on system:index (exact same scene ID)
join_filter = ee.Filter.equals(leftField="system:index", rightField="system:index")
joined      = ee.Join.inner().apply(colS2A, colS2_cloud, join_filter)

def match_collections(feature):
    """Merge each S2 image with its cloud probability band."""
    primary   = ee.Image(feature.get("primary"))
    secondary = ee.Image(feature.get("secondary"))
    return (primary
            .addBands(secondary.select("probability").rename('cloud_probability'))
            .copyProperties(primary, primary.propertyNames()))

colS2A_cloud = ee.ImageCollection(joined.map(match_collections))

In [ ]:
# ── 7b. Filter images with > 20% average cloud cover ─────────────────────────
# The cloud score is the mean cloud probability over the image footprint,
# computed at 200 m resolution for efficiency.
def addCloudScore(image):
    cloudScore = (image.select("cloud_probability")
                       .reduceRegion(reducer=ee.Reducer.mean(),
                                     geometry=image.geometry(),
                                     scale=200)
                       .get("cloud_probability"))
    return image.set("cloud_score", cloudScore).copyProperties(image)

colS2A_cloud_scored = colS2A_cloud.map(addCloudScore)
colS2A_filtered     = ee.ImageCollection(
    colS2A_cloud_scored.filter(ee.Filter.lt("cloud_score", 20)))

In [ ]:
# ── 7c. Pixel-level cloud probability mask (< 20%) ───────────────────────────
def maskClouds(image):
    """Mask pixels where cloud probability >= 20%."""
    cloud_mask = image.select('cloud_probability').lt(20)
    return image.updateMask(cloud_mask).copyProperties(image)

colS2A_cloud_masked = colS2A_filtered.map(maskClouds)

In [ ]:
# ── 7d. SCL mask: remove cloud shadows, thick/thin clouds, snow, saturated ───
# SCL classes masked out:
#   1 = saturated/defective   3 = cloud shadow
#   8 = medium probability cloud   9 = high probability cloud
#   10 = thin cirrus           11 = snow/ice
SCL_INVALID = [1, 3, 8, 9, 10, 11]

def maskSCL(image):
    """Mask pixels with invalid SCL classification."""
    scl   = image.select('SCL')
    valid = scl.neq(SCL_INVALID[0])
    for val in SCL_INVALID[1:]:
        valid = valid.And(scl.neq(val))
    return image.updateMask(valid).copyProperties(image, image.propertyNames())

# Apply both masks sequentially
colS2A_clean = colS2A_cloud_masked.map(maskSCL)
print("Cloud masking applied (S2_CLOUD_PROBABILITY + SCL).")

## Step 8 – Export list of source granules *(optional)*

Exports a CSV listing all Sentinel-2 granule IDs used (after cloud filtering).
Useful for metadata documentation.

In [ ]:
# This export is optional. It produces a CSV with one row per granule used.
# The file is saved in the same Drive folder as the raster outputs.
task_granules = ee.batch.Export.table.toDrive(
    collection=colS2A_clean.map(
        lambda img: ee.Feature(None, {'GRANULE_ID': img.get('GRANULE_ID')})),
    description=(DRIVE_FOLDER + '_export_S2_GRANULE_IDs_'
                 + remove_special_characters(tiles[0])
                 + '_' + startDate + '_' + endDate),
    folder=DRIVE_FOLDER,
    fileFormat='CSV'
)
task_granules.start()
print(f"Granule export task started (ID: {task_granules.id})")
print("Monitor at: https://code.earthengine.google.com/tasks")

## Step 9 – Harmonise band types

Convert all bands to Uint16 for consistency (the `cloud_probability` band is
originally stored as Byte). The `reproject` is intentionally omitted here —
GEE handles reprojection efficiently at export time.

In [ ]:
def harmonize_to_uint16(image):
    """Cast all spectral bands and cloud_probability to Uint16."""
    cloud  = image.select('cloud_probability').toUint16()
    others = image.select(['B2', 'B3', 'B4', 'B8', 'B8A', 'B11', 'B12']).toUint16()
    return others.addBands(cloud)
    # Note: reprojection is deferred to export (more efficient)

colS2A_final = colS2A_clean.map(harmonize_to_uint16)
print("Band harmonisation applied.")

## Step 10 – Compute spectral indices (NDVI and MNDWI)

In [ ]:
# ── NDVI : Normalised Difference Vegetation Index ────────────────────────────
# NDVI = (B8 - B4) / (B8 + B4)
# Range: -1 to 1. High values (> ~0.3) indicate dense vegetation.
# Negative values are physically valid (water, snow, bare soil).
def calcul_ndvi(image):
    return (image.normalizedDifference(['B8', 'B4'])
                 .rename('ndvi')
                 .copyProperties(image, image.propertyNames()))

colS2A_ndvi = colS2A_final.map(calcul_ndvi)

# ── MNDWI : Modified Normalised Difference Water Index ───────────────────────
# MNDWI = (Green - SWIR1) / (Green + SWIR1)  using B3 (Green) and B11 (SWIR1)
# High values indicate open water surfaces.
def mndwi_s2(image):
    swir  = image.select("B11")
    green = image.select("B3")
    return (green.subtract(swir)
                 .divide(green.add(swir))
                 .rename("mndwi")
                 .copyProperties(image, image.propertyNames()))

colS2A_mndwi = colS2A_final.map(mndwi_s2)
print("NDVI and MNDWI computed.")

## Step 11 – Edge pixel removal (feather mask)

Sentinel-2 tiles have unreliable pixels near their edges (radiometric
inconsistencies, partial overlap artefacts). This mask removes pixels within
`dist_th` pixels of the valid data boundary of each image.

The mask is computed from the **actual valid pixel boundary** (not the
rectangular tile footprint), which avoids visible seams at tile overlaps.

In [ ]:
def feather_mask(img):
    """
    Mask pixels too close to the edge of valid data within each S2 image.
    Uses the actual valid-pixel boundary (img.mask()) rather than the tile
    geometry, so tile-overlap seams are minimised in the final composite.

    dist_th (set per site in Step 3) controls the buffer width in pixels.
    """
    # Build a binary mask of valid pixels (1 = valid, 0 = masked/nodata)
    footprint = img.mask().reduce(ee.Reducer.min())

    # Compute Euclidean distance (in pixels) from the nearest invalid pixel
    distance = footprint.Not().fastDistanceTransform(30).sqrt()

    # Keep only pixels at least dist_th pixels away from any invalid pixel
    safe_mask = distance.gt(dist_th)
    return img.updateMask(safe_mask).copyProperties(img, img.propertyNames())

colS2A_ndvi_corr  = colS2A_ndvi.map(feather_mask)
colS2A_mndwi_corr = colS2A_mndwi.map(feather_mask)
print(f"Edge mask applied (dist_th = {dist_th} px).")

## Step 12 – Compute monthly mean NDVI composites

For each calendar month (1–12), compute the pixel-wise mean NDVI across all
valid observations over 2017–2024.

Pixels with no valid observation for a given month (due to persistent cloud
cover or insufficient satellite coverage) are filled with the **annual mean
composite** (average over all months and years). This ensures full spatial
coverage in the output.

In [ ]:
# ── Annual composite (used as fallback for missing pixels) ───────────────────
annual_composite = (colS2A_ndvi_corr
                    .select('ndvi')
                    .mean()
                    .toFloat()
                    .rename('mean_ndvi'))

# ── Monthly mean function ─────────────────────────────────────────────────────
months = ee.List.sequence(1, 12)

def _monthly_mean(m):
    """
    Compute the mean NDVI for calendar month m across all years (2017–2024).
    If no valid pixels exist for month m, returns a fully masked image.
    Missing pixels are filled with the annual composite.
    """
    m  = ee.Number(m)
    ic = (colS2A_ndvi_corr
          .filter(ee.Filter.calendarRange(m, m, 'month'))
          .select('ndvi'))

    mean_img = ic.mean()

    # Guard against completely empty months (no images at all)
    safe_img = ee.Image(ee.Algorithms.If(
        ic.size().gt(0),
        mean_img,
        ee.Image.constant(0).rename('ndvi').updateMask(ee.Image(0))
    ))

    # Fill residual masked pixels (cloud gaps, edge masks) with annual composite
    return (safe_img
            .toFloat()
            .rename('mean_ndvi')
            .unmask(annual_composite)
            .set({'month': m}))

monthly_means = ee.ImageCollection.fromImages(months.map(_monthly_mean))
print("Monthly composites computed (server-side, not yet exported).")

## Step 13 – Compute full-period composites (NDVI and MNDWI) *(optional)*

In [ ]:
# Full-period (2017–2024) mean NDVI — single image
vegetation_composite = (ee.Image(colS2A_ndvi_corr.select(['ndvi']).mean())
                        .toFloat()
                        .rename(['mean_ndvi'])
                        .clip(geometry_to_export))

# Full-period (2017–2024) mean MNDWI — single image
water_composite = (ee.Image(colS2A_mndwi_corr.select(['mndwi']).mean())
                   .toFloat()
                   .rename(['mean_mndwi'])
                   .clip(geometry_to_export))

print("Full-period composites computed.")

## Step 14 – Export to Google Drive

All exports go to the `R4Cs_WP6_layers` folder in your Google Drive.  
Monitor export progress at: https://code.earthengine.google.com/tasks

Output files are named: `{tile}_{site}_vegetation_{month:02d}.tif`  
(e.g. `CuronianlLagoon_vegetation_01.tif` for January)

In [ ]:
def export_image(image, name, folder, roi):
    """
    Export a single image to Google Drive as a GeoTIFF at 10 m resolution.
    The CRS is inherited from the image (native Sentinel-2 tile projection).
    """
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=name,
        folder=folder,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True},  # smaller files, faster QGIS loading
        scale=10,
        region=roi,
        maxPixels=1e13
    )
    task.start()
    print(f"  Task started: {name}  (ID: {task.id})")

In [ ]:
# ── Loop over each spatial tile and export selected products ─────────────────
features_list = geometry_to_export.toList(geometry_to_export.size())

for i in range(geometry_to_export.size().getInfo()):
    feature = ee.Feature(features_list.get(i))
    geom    = feature.geometry()
    tile    = str(feature.get("ID").getInfo())
    print(f"\nProcessing tile: {tile}")

    # Monthly mean NDVI composites (12 GeoTIFFs) — main output
    if do_month:
        print("Exporting monthly composites...")
        for m in range(1, 13):
            monthly_img = ee.Image(
                monthly_means.filter(ee.Filter.eq('month', m)).first()
            )
            monthly_img = (monthly_img
                           .reproject(crs=dst_crs, scale=10)
                           .clip(geom))
            out_name = f"{remove_special_characters(pilote)}_vegetation_{m:02d}"
            export_image(monthly_img, out_name, DRIVE_FOLDER, geom)

    # Full-period mean NDVI (single image)
    if do_mean_NDVI:
        export_image(vegetation_composite,
                     f"{tile}_{startDate}_{endDate}_mean_ndvi",
                     DRIVE_FOLDER, geom)

    # Full-period mean MNDWI (single image)
    if do_mean_MNDWI:
        export_image(water_composite,
                     f"{tile}_{startDate}_{endDate}_mean_mndwi",
                     DRIVE_FOLDER, geom)

    # All individual NDVI images (one per scene — large export)
    if do_NDVI:
        n = colS2A_ndvi.size().getInfo()
        col_list = colS2A_ndvi.toList(n)
        for j in range(n):
            img  = ee.Image(col_list.get(j))
            name = img.get('GRANULE_ID').getInfo() + '_ndvi'
            export_image(img, name, DRIVE_FOLDER, geom)

print("\nAll export tasks submitted.")
print("Monitor at: https://code.earthengine.google.com/tasks")